# Detectron2 Use Models 使用模型完整 Demo

文档来源：Detectron2 0.6 Documentation → Use Models

覆盖内容：
1. 从 Yacs 配置构建模型 build_model
2. 加载/保存检查点 DetectionCheckpointer
3. 模型调用：训练模式、推理模式、DefaultPredictor
4. Model Input Format 模型输入格式
5. Model Output Format 模型输出格式
6. 部分执行模型，获取中间特征

## 环境检测

运行前先检查本机的 Python / PyTorch / CUDA / detectron2 版本，确认环境匹配（CUDA 与 PyTorch、detectron2 wheel 需对应）。

In [ ]:
import sys
print("Python:       ", sys.version.split()[0])

try:
    import torch
    print("PyTorch:      ", torch.__version__)
    print("CUDA (torch): ", torch.version.cuda)
    print("GPU available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch:       未安装")

try:
    import detectron2
    print("detectron2:   ", detectron2.__version__)
except ImportError:
    print("detectron2:    未安装")

## 0. 环境准备

【Colab 取消注释执行】安装 detectron2，注意匹配 cuda、torch 版本。以下命令先注释保留，确认上一节环境检测无误后再按需执行。

In [ ]:
# !pip install detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu118/torch2.0/index.html

import torch
import numpy as np
from detectron2.config import get_cfg
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.utils.events import EventStorage
from detectron2.data import MetadataCatalog
from detectron2.structures import Instances, Boxes, ImageList
from detectron2.predictor import DefaultPredictor

## 1. 构建模型：基于 Yacs Config

`build_model` 只搭建网络结构，参数随机初始化

In [ ]:
# 获取默认配置，选用 Faster R-CNN 示例配置
cfg = get_cfg()
cfg.merge_from_file("detectron2/configs/COCO-Detection/faster_rcnn_R_50_FPN_1x.yaml")

# 构建模型，返回 torch.nn.Module
model = build_model(cfg)
print(f"✅模型构建完成，model type: {type(model)}")

## 2. Load / Save Checkpoint 加载保存权重

`DetectionCheckpointer` 负责 pth/pkl 权重加载与保存

In [ ]:
# 加载权重，一般读取 cfg.MODEL.WEIGHTS 指定的路径/url
checkpointer = DetectionCheckpointer(model)
# 实际使用填入权重文件路径或者 model zoo 链接：
# checkpointer.load(cfg.MODEL.WEIGHTS)

# 实例化 checkpointer，指定输出保存目录
checkpointer = DetectionCheckpointer(model, save_dir="./output")
# 保存模型到 ./output/model_999.pth
checkpointer.save("model_999")
print("✅模型已经保存到 ./output/model_999.pth")

## 3. 使用模型

模型输入为 `list[dict]`，每个 dict 代表一张图片

In [ ]:
# 构造模拟输入数据（推理模式输入）
# 推理仅需要 "image" key；image 张量格式 (C,H,W)，detectron2 默认 BGR
H_im, W_im = 480, 640
image_tensor = torch.randn(3, H_im, W_im)

# inputs 是 list[dict]，batch 维度由 list 长度表示
inputs = [
    {
        "image": image_tensor,
        "height": H_im,   # 期望输出分辨率（原图尺寸）
        "width": W_im
    }
]

### 3.1 推理模式 `model.eval()` + `torch.no_grad()`

In [ ]:
model.eval()
with torch.no_grad():
    outputs = model(inputs)

# outputs: list[dict]，每张图像对应一个 dict
print(f"✅推理完成，outputs 长度(batch size)：{len(outputs)}")
print(f"推理输出 keys: {list(outputs[0].keys())}")

### 3.2 训练模式：必须配合 EventStorage

训练输入需要带上 GT 标注信息 `instances`

In [ ]:
# 构造训练用的 instances 标注对象
gt_instances = Instances((H_im, W_im))
# 真实框 N=2
gt_instances.gt_boxes = Boxes(torch.tensor([[20, 20, 100, 120], [200, 100, 300, 220]]))
gt_instances.gt_classes = torch.tensor([0, 1], dtype=torch.long)

train_inputs = [
    {
        "image": torch.randn(3, H_im, W_im),
        "instances": gt_instances
    }
]

model.train()
# 训练模式必须在 EventStorage 上下文内运行
with EventStorage() as storage:
    loss_dict = model(train_inputs)

print(f"✅训练 loss 字典: {loss_dict}")

### 3.3 DefaultPredictor 高层封装，单图推理

封装预处理与模型加载，直接传入 OpenCV BGR 图像（numpy 数组）

In [ ]:
# predictor = DefaultPredictor(cfg)
# im_bgr = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
# pred_out = predictor(im_bgr)
# print("DefaultPredictor 输出 keys", pred_out.keys())

## 4. Model Input Format 输入格式说明

内置模型统一输入：`list[dict]`

| key | 说明 |
|-----|------|
| image | Tensor(C,H,W) BGR，受 cfg.INPUT.FORMAT 控制 |
| height,width | 输出期望分辨率，不一定等于 image 尺寸 |
| instances | 训练用 GT 标注 Instances 对象 |
| sem_seg | (H,W) 语义分割 GT |
| proposals | 候选框 Instances 对象 |

## 5. Model Output Format 输出格式

- train：返回 `dict[str -> ScalarTensor]` loss 字典
- eval：返回 `list[dict]`，每图一个 dict

eval 输出常用字段：
- instances：pred_boxes、scores、pred_classes、pred_masks、pred_keypoints
- sem_seg：(num_classes, H, W)
- proposals：proposal_boxes, objectness_logits
- panoptic_seg：(pred_tensor, segments_info)

## 6. Partially execute a model：部分执行模型，提取中间特征

不调用完整 forward，手动逐层跑网络，拿到中间层输出

In [ ]:
model.eval()
# 构造 ImageList，封装 batch 图像 tensor
image_list = ImageList.from_tensors([torch.randn(3, 480, 640)])

with torch.no_grad():
    # backbone 前向，得到多尺度特征
    features = model.backbone(image_list.tensor)
    # proposal generator 生成候选框
    proposals, _ = model.proposal_generator(image_list, features)
    # roi_heads 得到检测实例
    instances, _ = model.roi_heads(image_list, features, proposals)
    # 获取 mask head 之前的 mask 特征
    mask_features = [features[f] for f in model.roi_heads.in_features]
    mask_features = model.roi_heads.mask_pooler(mask_features, [x.pred_boxes for x in instances])

print(f"✅提取中间 mask_features shape: {mask_features.shape}")